In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [ ]:
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(llm_schema)

llm_structured_output.invoke("The movie was funny")


llm_schema(movie_summary_flag='positive')

# **Chain with Conditional Chains**

In [5]:
#Task 1 [Prompt]
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a movie review evaluator"),
    ("human", "Please categorize the review as positive or negative: {input}")
])

#Task 2 [LLm]
llm_structured_output = llm_openai.with_structured_output(llm_schema)

In [3]:
#Task 3 [Custom Runnable]
from langchain_core.runnables import RunnableLambda

def pydantic_json(input: llm_schema) -> str:
    return input.model_dump()["movie_summary_flag"]

#create a wrapper for the function
pydantic_json_lambda = RunnableLambda(pydantic_json)

# **Option 1: Decicated chains**

In [8]:
# Task 1 [Prompt]
linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")
])

#Task 2 [LLm]
llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

#Task 3 [Str Parser]
str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm_openai | str_parser

# **Option 2: function**

In [16]:
def insta_chain(text: str):

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a Instagram post generator"),
        ("human", "Create a post for the following text for Instagram: {text}")
    ])

    #Task 2 [LLm]
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0)

    #Task 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke({"text": text})
    return result

#it's easy to manage

insta_chain_runnable = RunnableLambda(insta_chain)

# **Final Orchestation**

In [17]:
from langchain_core.runnables import RunnableBranch


contitional_chain = RunnableBranch(
    (lambda x : "positive" in x, chain_linkedin),
    insta_chain_runnable
)

In [18]:
final_chain = (
    prompt_template |
    llm_structured_output |
    pydantic_json_lambda | 
    contitional_chain
)

final_chain.invoke({"input" : "The movie was really bad"})

'Here are ready-to-use Instagram post options based on the single word: "negative." Pick one caption + image concept, or mix-and-match.\n\nMain post (caption + image idea)\n- Caption: "Not every negative is a setback — sometimes it’s your doorway. See what you can learn, then step forward. ✨"\n- Image idea: Moody photo of a doorway, a shadowed street, or a film negative held up to light.\n- Hashtags: #Perspective #GrowthMindset #TurnItAround #NegativeToPositive #LearnAndGrow\n- Alt text: "A shadowed doorway with sunlight spilling through, symbolizing leaving negativity behind."\n- CTA: "Double-tap if you’re choosing growth — share one lesson you’ve learned from a negative experience."\n\nAlternative captions (choose tone)\n- Reflective: "Negative moments are temporary. The lesson is permanent. 🌱"\n- Bold: "Negative vibes? Set the boundary. Protect your peace. ✋"\n- Playful: "Caught a case of the negatives — developing into something better. 🎞️"\n- Minimal: "Negative → Perspective → Pow